# 7 · Widgets

A widget is the control a person edits an input with. Every input names its own; no type has a default.

In [ ]:
import json
from typing import Annotated, Literal

from conductor import Choice, NodeRegistry, Param, Series, Table, make_graph, node
from conductor.widgets import DatePicker, Dropdown, FileUpload, ListInput, NumberInput, Slider, Switch, Textarea, TextInput, Widget

## Widgets with constraints

The widget goes on the `Param`, beside the `title`, the `description` and the limits a value must keep.

In [ ]:
PRIORITIES = (
    Choice(id="low", title="Low"),
    Choice(id="normal", title="Normal"),
    Choice(id="high", title="High"),
)


@node("send-email", title="Send Email", description="Send an email", category="mail")
def send_email(
    to: Annotated[str, Param(title="Recipient", pattern=r"^[^@]+@[^@]+$", widget=TextInput())],
    subject: Annotated[str, Param(max_length=200, widget=TextInput())],
    body: Annotated[str, Param(min_length=10, widget=Textarea(rows=8))],
    priority: Annotated[str, Param(choices=PRIORITIES, widget=Dropdown())] = "normal",
    retries: Annotated[int, Param(title="Retry attempts", ge=0, le=5, widget=Slider(step=1))] = 1,
) -> Annotated[str, Param(title="Status")]:
    return f"sent: {to=}, {subject=}, {priority=}, {retries=}"


for field in send_email.describe().inputs:
    kind = field.widget["type"] if field.widget else "none"
    print(f"  {field.name:<10} type={kind:<10} title={field.title!r}  schema={field.json_schema}  required={field.required}")

## Every widget, and one of your own

An input may be any type pydantic can validate. A `Series[X]` input takes `ListInput`, or no widget when an edge fills it.

The limits a value must keep to are on the `Param`, in pydantic's words, and they are enforced. A control conductor does not ship is a `Widget` subclass in your own code, next to the component that renders it.

In [ ]:
class EntityDropdown(Widget):
    """Choices the host resolves. Conductor carries it to the editor and never reads it."""

    type: Literal["entity-dropdown"] = "entity-dropdown"
    entity_kind: str = ""


@node("widget-showcase", title="Showcase", description="Every widget", category="demo")
def showcase(
    # Text
    heading: Annotated[str, Param(title="Title", max_length=100, widget=TextInput())],
    notes: Annotated[str, Param(widget=Textarea(rows=6))],
    # Choice
    style: Annotated[str, Param(choices=(Choice(id="a", title="A"), Choice(id="b", title="B")), widget=Dropdown())],
    user: Annotated[str, Param(widget=EntityDropdown(entity_kind="user"))],
    # Numeric
    count: Annotated[int, Param(ge=0, widget=NumberInput())],
    volume: Annotated[float, Param(ge=0, le=1, widget=Slider(step=0.01))],
    # Boolean
    enabled: Annotated[bool, Param(widget=Switch())],
    # Date & file
    starts: Annotated[str, Param(widget=DatePicker(initial="today"))],
    upload: Annotated[str, Param(widget=FileUpload(accept=(".pdf", ".csv"), max_size_mb=10))],
    # Collections
    items: Annotated[Series[str], Param(widget=ListInput())],
    sources: Series[str],
    # Structured
    table: Table,
    config: Annotated[dict, Param(widget=Textarea())],
) -> Annotated[str, Param(title="Summary")]:
    return "all good"


inputs = showcase.describe().inputs
print(f"{len(inputs)} inputs, {len({field.widget['type'] for field in inputs if field.widget})} distinct widget kinds:\n")
for field in inputs:
    kind = field.widget["type"] if field.widget else "none"
    print(f"  {field.name:<13} {field.type or 'table':<8} widget={kind}")

## The dump

A widget dumps through pydantic, its own fields and `type`, for an editor to read.


In [ ]:
w = Slider(step=5)
print(json.dumps(w.model_dump(mode="json"), indent=2))


In [ ]:
# The same widget as the field's description carries it: the title and the limits sit beside it.
volume = next(field for field in inputs if field.name == "volume")
print(json.dumps(volume.model_dump(mode="json", exclude_none=True), indent=2))

## Running a graph

A typed-in value is checked against its input's type.

In [ ]:
registry = NodeRegistry()


@registry.node("label", title="Label", description="Glue", category="demo")
def label(
    n: Annotated[int, Param(title="N", widget=NumberInput())],
    name: Annotated[str, Param(title="Label", widget=TextInput())],
) -> Annotated[str, Param(title="Out")]:
    return f"{name}={n}"


graph = registry.load(make_graph(label, values={"label.n": 42, "label.name": "answer"}))
graph.run().results["label"]["result"]